In [1]:
!pip install mwclient
import mwclient
site = mwclient.Site('en.wikipedia.org')
page = site.pages['Bitcoin']
page

<Page object 'Bitcoin' for <Site object 'en.wikipedia.org/w/'>>

In [2]:
revs = list(page.revisions())

In [3]:
revs[0]

OrderedDict([('revid', 1281323903),
             ('parentid', 1280330283),
             ('minor', ''),
             ('user', 'Davide King'),
             ('timestamp',
              time.struct_time(tm_year=2025, tm_mon=3, tm_mday=19, tm_hour=17, tm_min=46, tm_sec=48, tm_wday=2, tm_yday=78, tm_isdst=-1)),
             ('comment', '/* 2020–present */ copyedit')])

In [4]:
revs = sorted(revs,key=lambda revs:revs['timestamp'])

In [5]:
revs[0]
# revs[0]['timestamp']

OrderedDict([('revid', 275832581),
             ('parentid', 0),
             ('user', 'Pratyeka'),
             ('timestamp',
              time.struct_time(tm_year=2009, tm_mon=3, tm_mday=8, tm_hour=16, tm_min=41, tm_sec=7, tm_wday=6, tm_yday=67, tm_isdst=-1)),
             ('comment', 'creation (stub)')])

In [6]:
len(revs)

17984

In [7]:
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis")

# result = sentiment_pipeline(["I love this product!"])
# print(result)

def find_sentiment(text):
    sent = sentiment_pipeline([text[:250]])[0]
    score = sent["score"]
    if sent["label"] == "NEGATIVE":
        score *= -1
    return score


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Why Use [0]?
Since the result is a list with one element, we need to extract the first element to access the dictionary inside.


```
sent = sentiment_pipeline(["I love this product!"])[0]  # Extract the first element
print(sent)

{'label': 'POSITIVE', 'score': 0.9998}
```


Now, we have only the dictionary instead of a list.

In [8]:
find_sentiment("i lavis you")

-0.8655312061309814

just for measuring metrics evaltion

```
from sklearn.metrics import r2_score
texts = [
    "I love this product!",       # Expected sentiment: 0.9 (Very positive)
    "This is a terrible service!", # Expected sentiment: -0.8 (Very negative)
    "It's okay, not great.",       # Expected sentiment: 0.2 (Neutral)
    "Absolutely fantastic!",       # Expected sentiment: 1.0 (Highly positive)
    "Worst experience ever."       # Expected sentiment: -1.0 (Highly negative)
]

actual_score = [0.9, -0.8, 0.2, 1.0, -1.0]
predicted_score = [find_sentiment(text) for text in texts]

r2 = r2_score(actual_score,predicted_score)
print("Actual Scores:   ", actual_score)
print("Predicted Scores:", predicted_score)
print("R² Score:        ", r2)
```
the score coming is 0.5

In [9]:
import time
edits = {}

for rev in revs:
    date = time.strftime("%Y-%m-%d", rev["timestamp"])
    if date not in edits:
        edits[date] = dict(sentiments=list(), edit_count=0)

    edits[date]["edit_count"] += 1

    comment = rev.get("comment", "")
    edits[date]["sentiments"].append(find_sentiment(comment))
edits

{'2009-03-08': {'sentiments': [-0.9905919432640076,
   0.7481208443641663,
   -0.9907428622245789,
   -0.9688863158226013],
  'edit_count': 4},
 '2009-08-05': {'sentiments': [0.7481208443641663], 'edit_count': 1},
 '2009-08-06': {'sentiments': [0.9957457184791565, 0.9957457184791565],
  'edit_count': 2},
 '2009-08-14': {'sentiments': [0.930020809173584], 'edit_count': 1},
 '2009-10-13': {'sentiments': [0.5404348969459534, -0.9954361319541931],
  'edit_count': 2},
 '2009-11-18': {'sentiments': [0.8839507699012756], 'edit_count': 1},
 '2009-12-08': {'sentiments': [-0.9869275689125061], 'edit_count': 1},
 '2009-12-17': {'sentiments': [-0.9975171089172363], 'edit_count': 1},
 '2010-02-23': {'sentiments': [-0.9994946718215942], 'edit_count': 1},
 '2010-03-18': {'sentiments': [0.875877320766449], 'edit_count': 1},
 '2010-04-13': {'sentiments': [0.930020809173584,
   0.815800666809082,
   0.815800666809082,
   0.815800666809082],
  'edit_count': 4},
 '2010-04-15': {'sentiments': [0.9300208091

In [10]:
from statistics import mean

for keys in edits:
  if len(edits[keys]['sentiments'])> 0:
    edits[keys]['mean_sentiments'] = mean(edits[keys]['sentiments'])
    edits[keys]["neg_sentiments"] = len([s for s in edits[keys]['sentiments'] if s < 0]) / len(edits[keys]['sentiments'])
    edits[keys]["pos_sentiments"] = len([s for s in edits[keys]['sentiments'] if s > 0]) / len(edits[keys]['sentiments'])

  else:
    edits[keys]['mean_sentiments'] = 0
    edits[keys]['neg_sentiments'] = 0
    edits[keys]['pos_sentiments'] = 0
  del edits[keys]['sentiments']
edits

{'2009-03-08': {'edit_count': 4,
  'mean_sentiments': -0.5505250692367554,
  'neg_sentiments': 0.75,
  'pos_sentiments': 0.25},
 '2009-08-05': {'edit_count': 1,
  'mean_sentiments': 0.7481208443641663,
  'neg_sentiments': 0.0,
  'pos_sentiments': 1.0},
 '2009-08-06': {'edit_count': 2,
  'mean_sentiments': 0.9957457184791565,
  'neg_sentiments': 0.0,
  'pos_sentiments': 1.0},
 '2009-08-14': {'edit_count': 1,
  'mean_sentiments': 0.930020809173584,
  'neg_sentiments': 0.0,
  'pos_sentiments': 1.0},
 '2009-10-13': {'edit_count': 2,
  'mean_sentiments': -0.22750061750411987,
  'neg_sentiments': 0.5,
  'pos_sentiments': 0.5},
 '2009-11-18': {'edit_count': 1,
  'mean_sentiments': 0.8839507699012756,
  'neg_sentiments': 0.0,
  'pos_sentiments': 1.0},
 '2009-12-08': {'edit_count': 1,
  'mean_sentiments': -0.9869275689125061,
  'neg_sentiments': 1.0,
  'pos_sentiments': 0.0},
 '2009-12-17': {'edit_count': 1,
  'mean_sentiments': -0.9975171089172363,
  'neg_sentiments': 1.0,
  'pos_sentiments': 

In [11]:
# Creating a DataFrame from a dictionary.
import pandas as pd

edits_df = pd.DataFrame.from_dict(edits,orient ='index')
edits_df

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2009-03-08,4,-0.550525,0.75,0.25
2009-08-05,1,0.748121,0.00,1.00
2009-08-06,2,0.995746,0.00,1.00
2009-08-14,1,0.930021,0.00,1.00
2009-10-13,2,-0.227501,0.50,0.50
...,...,...,...,...
2025-03-10,1,0.964989,0.00,1.00
2025-03-11,1,-0.857484,1.00,0.00
2025-03-12,1,-0.996866,1.00,0.00
2025-03-13,2,-0.996552,1.00,0.00


```
orient="index" in Pandas
In Pandas, the orient="index" parameter is used when converting a dictionary to a DataFrame using pd.DataFrame.from_dict().

This tells Pandas that the keys of the dictionary should become the row labels (index) of the DataFrame instead of column labels.
```

In [12]:
# Converts the index (dates) to a proper datetime format for time-based operations.
edits_df.index = pd.to_datetime(edits_df.index)
edits_df

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2009-03-08,4,-0.550525,0.75,0.25
2009-08-05,1,0.748121,0.00,1.00
2009-08-06,2,0.995746,0.00,1.00
2009-08-14,1,0.930021,0.00,1.00
2009-10-13,2,-0.227501,0.50,0.50
...,...,...,...,...
2025-03-10,1,0.964989,0.00,1.00
2025-03-11,1,-0.857484,1.00,0.00
2025-03-12,1,-0.996866,1.00,0.00
2025-03-13,2,-0.996552,1.00,0.00


In [13]:
from datetime import datetime
date = pd.date_range(start='2009-03-08',end=datetime.today())
edits_df = edits_df.reindex(date,fill_value=0)
edits_df

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2009-03-08,4,-0.550525,0.75,0.25
2009-03-09,0,0.000000,0.00,0.00
2009-03-10,0,0.000000,0.00,0.00
2009-03-11,0,0.000000,0.00,0.00
2009-03-12,0,0.000000,0.00,0.00
...,...,...,...,...
2025-03-27,0,0.000000,0.00,0.00
2025-03-28,0,0.000000,0.00,0.00
2025-03-29,0,0.000000,0.00,0.00
2025-03-30,0,0.000000,0.00,0.00


In [14]:
# Compute 30-Day Rolling Average
rolling_edits = edits_df.rolling(30, min_periods=30).mean()
rolling_edits = rolling_edits.dropna()
rolling_edits

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2009-04-06,0.133333,-0.018351,0.025,0.008333
2009-04-07,0.000000,0.000000,0.000,0.000000
2009-04-08,0.000000,0.000000,0.000,0.000000
2009-04-09,0.000000,0.000000,0.000,0.000000
2009-04-10,0.000000,0.000000,0.000,0.000000
...,...,...,...,...
2025-03-27,0.300000,-0.161421,0.200,0.033333
2025-03-28,0.300000,-0.161421,0.200,0.033333
2025-03-29,0.300000,-0.161421,0.200,0.033333
2025-03-30,0.300000,-0.161421,0.200,0.033333


In [15]:
rolling_edits.to_csv("btc_sent_roll.csv")